# AND-105 Task 8: Evaluation of LLM-Generated Risk Explanations

**Model under review:** `qwen2.5:1.5b` (Ollama, CPU-only inference, ~10–29 s/call)  
**Evaluation sample:** 10 HIGH-risk elevators, risk_score = 1.0  
**Explanations generated by:** `intelligence/generate_explanations.py` (AND-105 Task 7, pre-extraction strategy)  
**Total explanations in production DB:** 5,015  

## Evaluation Dimensions

| Dimension | Question |
|---|---|
| **Accuracy** | Do the facts in the explanation match the source DB records? |
| **Usefulness** | Would it help a field technician know what to inspect? |
| **Consistency** | Are similar risk profiles described with consistent quality? |

In [1]:
import asyncio, sys
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

import json
import re
import subprocess
from collections import Counter, defaultdict
from datetime import datetime

import pandas as pd
from IPython.display import display, Markdown

CONTAINER = 'rocket-elevators-ops-dashboard-db-1'
DB_USER   = 'api_user'
DB_NAME   = 'rocket_elevators'


def query_json(sql):
    """Execute SQL inside the DB container via docker exec; return list of dicts."""
    wrapped = f'SELECT json_agg(row_to_json(t)) FROM ({sql}) t'
    result = subprocess.run(
        ['docker', 'exec', CONTAINER, 'psql', '-U', DB_USER, '-d', DB_NAME, '-t', '-A', '-c', wrapped],
        capture_output=True, text=True, encoding='utf-8',
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    raw = result.stdout.strip()
    return json.loads(raw) if raw else []


# Verify DB connection and count available explanations
test = query_json('SELECT count(*) AS n FROM predictions WHERE risk_explanation IS NOT NULL')
print(f'DB connected \u2713  explanations in DB: {test[0]["n"]:,}')

DB connected ✓  explanations in DB: 5,015


In [2]:
# ── Evaluation sample ─────────────────────────────────────────────────────────
# 10 specific elevators selected as the evaluation set.
# IDs derived from the first batch processed in the overnight Task 7 run.
SAMPLE_IDS = [9, 10, 119, 657, 851, 862, 863, 868, 869, 870]
id_list = ','.join(str(x) for x in SAMPLE_IDS)

# Fetch explanations + elevator metadata
rows = query_json(
    f'SELECT p.elevator_id, e.location, '
    f'COALESCE(e.elevator_type, \'Elevator\') AS elevator_type, '
    f'p.risk_level, p.risk_score::float8, p.risk_explanation '
    f'FROM predictions p '
    f'JOIN elevators e ON e.elevator_id = p.elevator_id '
    f'WHERE p.elevator_id IN ({id_list}) '
    f'ORDER BY p.elevator_id'
)

# Fetch all inspections for ground truth comparison
insp_rows = query_json(
    f'SELECT elevator_id, inspection_type, '
    f'latest_inspection_date::text AS date, outcome '
    f'FROM inspections '
    f'WHERE elevator_id IN ({id_list}) '
    f'ORDER BY elevator_id, latest_inspection_date DESC NULLS LAST'
)

# Build per-elevator ground truth
PASSING_OUTCOMES = {'Passed', 'Complete', 'All Orders Resolved'}
inspections_by_id = defaultdict(list)
for r in insp_rows:
    inspections_by_id[r['elevator_id']].append(r)

for e in rows:
    insp = inspections_by_id[e['elevator_id']]
    e['inspections'] = insp
    e['total_insp']  = len(insp)
    e['failed_insp'] = sum(1 for i in insp if i['outcome'] not in PASSING_OUTCOMES)

# Summary
print(f'Loaded {len(rows)} elevators | ground truth (inspections) attached')
print()
print(f'{"ID":>6}  {"Level":<6}  {"Score":>5}  {"Insp":>5}  {"Failed":>6}  Location')
print('-' * 70)
for e in rows:
    loc = e['location'][:36] + '..' if len(e['location']) > 36 else e['location']
    print(f"{e['elevator_id']:>6}  {e['risk_level']:<6}  {e['risk_score']:>5.1f}  "
          f"{e['total_insp']:>5}  {e['failed_insp']:>6}  {loc}")

Loaded 10 elevators | ground truth (inspections) attached

    ID  Level   Score   Insp  Failed  Location
----------------------------------------------------------------------
     9  HIGH      1.0      7       6  111 WELLESLEY ST W  TORONTO M7A 1A2 ..
    10  HIGH      1.0      9       6  111 WELLESLEY ST W  TORONTO M7A 1A2 ..
   119  HIGH      1.0      4       2  130 QUEEN ST W  TORONTO M5H 2N5 ON C..
   657  HIGH      1.0      5       3  259 ST. PAUL ST  ST CATHARINES L2R 3..
   851  HIGH      1.0      2       2  245 CARLAW AVE  TORONTO M4M 2S1 ON C..
   862  HIGH      1.0      3       1  27 THORNCLIFFE PARK DR  TORONTO M4H ..
   863  HIGH      1.0      4       3  27 THORNCLIFFE PARK DR  TORONTO M4H ..
   868  HIGH      1.0      2       0  171 NEPEAN ST  OTTAWA K2P 0B4 ON CA
   869  HIGH      1.0      5       3  335 COOPER ST  OTTAWA K2P 0G6 ON CA
   870  HIGH      1.0      6       5  335 COOPER ST  OTTAWA K2P 0G6 ON CA


## Evaluation Framework

### Accuracy Labels

| Label | Meaning |
|---|---|
| `fully_accurate` | All verifiable facts (dates, counts, outcomes) match source records exactly |
| `minor_inaccuracy` | Small factual error or missing key detail; overall direction is correct |
| `major_inaccuracy` | A core claim is significantly wrong (e.g., wrong time unit AND wrong count) |
| `hallucination` | Explanation contains information not present in source data |

### Usefulness Dimensions

| Dimension | Pass criterion |
|---|---|
| Identifies specific risk factors | Mentions at least one date, count, or outcome from the source |
| Helps technician | Actionable — tells the technician what to verify or look for |
| Is concise (2–3 sentences) | Not a single terse sentence; not verbose (4+) |

### Automated Heuristics (run before manual scoring)

Four checks applied programmatically:
1. **Sentence count** — expected 2–3
2. **Source date presence** — at least one inspection date from the source appears in the explanation (checks both ISO `YYYY-MM-DD` and text `Month D, YYYY` format; model was observed to output text format exclusively)
3. **Hedging language** — presence of phrases like 'may', 'could', 'Based on', 'if not addressed'
4. **Time unit error** — explanation says 'past N months' while source inspections span > 6 months

In [3]:
HEDGING_TERMS = [
    'may ', 'might ', 'could ', 'potentially ', 'possibly ',
    'if not addressed', 'based on', 'suggests a',
    'could lead to', 'indicates potential',
]
DATE_RE  = re.compile(r'\b\d{4}-\d{2}-\d{2}\b')
MONTH_RE = re.compile(r'past\s+\w+\s+month', re.IGNORECASE)


def count_sentences(text):
    return max(1, len(re.findall(r'[.!?]+(?:\s|$)', text.strip())))


def has_source_date(explanation, inspections):
    # Check ISO format (2015-03-27)
    exp_iso   = set(DATE_RE.findall(explanation))
    src_dates = {i['date'] for i in inspections if i.get('date')}
    if exp_iso & src_dates:
        return True
    # Model outputs text dates ("March 27, 2015"), not ISO — convert source dates and recheck
    for src_date in src_dates:
        try:
            dt = datetime.strptime(src_date, '%Y-%m-%d')
            text_date = f"{dt.strftime('%B')} {dt.day}, {dt.year}"
            if text_date in explanation:
                return True
        except ValueError:
            pass
    return False


def has_hedging(explanation):
    lo = explanation.lower()
    return any(t in lo for t in HEDGING_TERMS)


def time_unit_error(explanation, inspections):
    """True if explanation says 'past N months' but source spans > 6 months."""
    if not MONTH_RE.search(explanation):
        return False
    dates = [i['date'] for i in inspections if i.get('date')]
    if len(dates) < 2:
        return False
    parsed = sorted([datetime.strptime(d, '%Y-%m-%d') for d in dates])
    return (parsed[-1] - parsed[0]).days > 180


heuristic_rows = []
for e in rows:
    exp  = e.get('risk_explanation') or ''
    insp = e['inspections']
    heuristic_rows.append({
        'elevator_id':     e['elevator_id'],
        'sentences':       count_sentences(exp),
        'cites_src_date':  has_source_date(exp, insp),
        'has_hedging':     has_hedging(exp),
        'time_unit_error': time_unit_error(exp, insp),
    })

df_h = pd.DataFrame(heuristic_rows)
pd.set_option('display.max_colwidth', 60)
display(df_h.to_string(index=False))

print()
print(f'Avg sentence count:       {df_h.sentences.mean():.1f}   (target: 2–3)')
print(f'Cites source date:        {int(df_h.cites_src_date.sum())}/10  (note: model uses text dates, not ISO)')
print(f'Has hedging language:     {int(df_h.has_hedging.sum())}/10')
print(f'Time unit error:          {int(df_h.time_unit_error.sum())}/10')

' elevator_id  sentences  cites_src_date  has_hedging  time_unit_error\n           9          2            True        False            False\n          10          1           False        False            False\n         119          2           False        False             True\n         657          3            True         True            False\n         851          2           False         True            False\n         862          2           False        False            False\n         863          3            True         True            False\n         868          2            True         True            False\n         869          2            True         True            False\n         870          3            True        False            False'


Avg sentence count:       2.2   (target: 2–3)
Cites source date:        6/10  (note: model uses text dates, not ISO)
Has hedging language:     5/10
Time unit error:          1/10


## Accuracy Evaluation

Each explanation was compared line-by-line against the inspections table (ground truth).
Scores are pre-populated in the `ACCURACY` dictionary below based on observed findings;
the table re-generates automatically on every execution.

In [4]:
# Pre-populated accuracy scores — format: elevator_id -> (label, justification)
# Derived from manual comparison of explanation text vs inspections table.
ACCURACY = {
    9:   ('minor_inaccuracy',
          'States "four failed inspections in the past five years"; '
          'ground truth has 6 non-passing (2012–2015). '
          'Correct date cited (2015-03-27). Count off by 2.'),
    10:  ('minor_inaccuracy',
          'Single terse sentence with no dates, counts, or inspection types. '
          'Factually not wrong but not actionable. '
          'Most recent inspection (All Orders Resolved) not mentioned.'),
    119: ('major_inaccuracy',
          '"Four non-passing inspections over the past four months" — '
          'both wrong: source has 2 non-passing, and inspections span '
          '2012–2015 (>3 years, not months). Time unit and count both wrong.'),
    657: ('fully_accurate',
          'Correct date (2016-02-17), correct outcome (DC Follow up). '
          'Appropriate secondary context (no incidents, no pending alterations).'),
    851: ('minor_inaccuracy',
          'Count correct (2 non-passing). '
          'Hedging opener "Based on"; no specific dates cited.'),
    862: ('minor_inaccuracy',
          'Count correct (1 of 3 non-passing). '
          'Most recent inspection (2014-02-22 Complete) was passing — not mentioned. No dates.'),
    863: ('fully_accurate',
          'Correct date (2014-02-21), correct type (ED-Periodic), '
          'correct count (3 non-passing). Time span claim accurate.'),
    868: ('minor_inaccuracy',
          'Both inspections passed — explanation omits this. '
          'Circular phrasing: "no recent inspection since its most recent inspection". '
          'Correct date (2015-03-05).'),
    869: ('hallucination',
          '"resulted in minor corrections" not present in source data. '
          'Also states "no failed inspections since then" which contradicts '
          '2014-11-06 Follow up record (same date range).'),
    870: ('minor_inaccuracy',
          'Count off by one: claims 4 non-passing, source has 5. '
          'Correct date (2014-11-26), correct type (DC Follow up).'),
}

acc_rows = []
for e in rows:
    eid = e['elevator_id']
    label, justification = ACCURACY.get(eid, ('not_scored', 'N/A'))
    acc_rows.append({'elevator_id': eid, 'label': label, 'justification': justification})

df_acc = pd.DataFrame(acc_rows)
pd.set_option('display.max_colwidth', None)
display(df_acc)

print()
print('Distribution:')
for cat in ['fully_accurate', 'minor_inaccuracy', 'major_inaccuracy', 'hallucination']:
    n   = int((df_acc['label'] == cat).sum())
    pct = n / len(df_acc) * 100
    print(f'  {cat:<22} {n}  ({pct:.0f}%)')

,elevator_id,label,justification
0,9,minor_inaccuracy,"States ""four failed inspections in the past five years""; ground truth has 6 non-passing (2012–2015). Correct date cited (2015-03-27). Count off by 2."
1,10,minor_inaccuracy,"Single terse sentence with no dates, counts, or inspection types. Factually not wrong but not actionable. Most recent inspection (All Orders Resolved) not mentioned."
2,119,major_inaccuracy,"""Four non-passing inspections over the past four months"" — both wrong: source has 2 non-passing, and inspections span 2012–2015 (>3 years, not months). Time unit and count both wrong."
3,657,fully_accurate,"Correct date (2016-02-17), correct outcome (DC Follow up). Appropriate secondary context (no incidents, no pending alterations)."
4,851,minor_inaccuracy,"Count correct (2 non-passing). Hedging opener ""Based on""; no specific dates cited."
5,862,minor_inaccuracy,Count correct (1 of 3 non-passing). Most recent inspection (2014-02-22 Complete) was passing — not mentioned. No dates.
6,863,fully_accurate,"Correct date (2014-02-21), correct type (ED-Periodic), correct count (3 non-passing). Time span claim accurate."
7,868,minor_inaccuracy,"Both inspections passed — explanation omits this. Circular phrasing: ""no recent inspection since its most recent inspection"". Correct date (2015-03-05)."
8,869,hallucination,"""resulted in minor corrections"" not present in source data. Also states ""no failed inspections since then"" which contradicts 2014-11-06 Follow up record (same date range)."
9,870,minor_inaccuracy,"Count off by one: claims 4 non-passing, source has 5. Correct date (2014-11-26), correct type (DC Follow up)."



Distribution:
  fully_accurate         2  (20%)
  minor_inaccuracy       6  (60%)
  major_inaccuracy       1  (10%)
  hallucination          1  (10%)


## Usefulness Evaluation

Three binary dimensions scored for each explanation:

- **Identifies specific risk factors** — at least one specific date, count, or outcome named
- **Helps technician** — output is actionable; tells the technician what to look for or verify
- **Is concise (2–3 sentences)** — single-sentence outputs are too sparse; 4+ are verbose

In [5]:
# Pre-populated usefulness scores
# Format: elevator_id -> (risk_factors, helps_tech, is_concise, note)
USEFULNESS = {
    9:   (True,  True,  True,  'Date and pattern described; 2 sentences; actionable'),
    10:  (False, False, False, '1 sentence; no dates, counts, or types; not actionable'),
    119: (True,  False, True,  'Names a count but both count and time unit are wrong; could mislead'),
    657: (True,  True,  True,  'Specific date + type; technician knows exactly which inspection to review'),
    851: (True,  True,  True,  'Count correct; hedging weakens confidence but direction is right'),
    862: (True,  True,  True,  'Count and proportion cited; lacks dates but still useful'),
    863: (True,  True,  True,  'Date + type + count all cited; most useful explanation in sample'),
    868: (True,  False, True,  'Correctly identifies gap since last inspection; omits that all inspections passed'),
    869: (True,  False, True,  'Hallucinated detail could direct technician to a non-issue'),
    870: (True,  True,  True,  'Pattern identified; count off by one but direction correct'),
}

use_rows = []
for e in rows:
    eid = e['elevator_id']
    rf, ht, concise, note = USEFULNESS.get(eid, (False, False, False, 'N/A'))
    use_rows.append({
        'elevator_id':           eid,
        'risk_factors':          '\u2713' if rf      else '\u2717',
        'helps_technician':      '\u2713' if ht      else '\u2717',
        'is_concise (2-3 sent)': '\u2713' if concise else '\u2717',
        'notes':                 note,
    })

df_use = pd.DataFrame(use_rows)
display(df_use)

print()
rf_n = sum(1 for e in rows if USEFULNESS.get(e['elevator_id'], (False,))[0])
ht_n = sum(1 for e in rows if USEFULNESS.get(e['elevator_id'], (False, False))[1])
c_n  = sum(1 for e in rows if USEFULNESS.get(e['elevator_id'], (False, False, False))[2])
print(f'Identifies risk factors:  {rf_n}/10')
print(f'Helps technician:         {ht_n}/10')
print(f'Is concise (2-3 sent):    {c_n}/10')

,elevator_id,risk_factors,helps_technician,is_concise (2-3 sent),notes
0,9,✓,✓,✓,Date and pattern described; 2 sentences; actionable
1,10,✗,✗,✗,"1 sentence; no dates, counts, or types; not actionable"
2,119,✓,✗,✓,Names a count but both count and time unit are wrong; could mislead
3,657,✓,✓,✓,Specific date + type; technician knows exactly which inspection to review
4,851,✓,✓,✓,Count correct; hedging weakens confidence but direction is right
5,862,✓,✓,✓,Count and proportion cited; lacks dates but still useful
6,863,✓,✓,✓,Date + type + count all cited; most useful explanation in sample
7,868,✓,✗,✓,Correctly identifies gap since last inspection; omits that all inspections passed
8,869,✓,✗,✓,Hallucinated detail could direct technician to a non-issue
9,870,✓,✓,✓,Pattern identified; count off by one but direction correct



Identifies risk factors:  9/10
Helps technician:         6/10
Is concise (2-3 sent):    9/10


## Consistency Evaluation

Three pairs of elevators with **similar risk profiles** are compared to assess whether
the model describes analogous situations with consistent quality and structure.

Pairs are selected based on:
- Same building address
- Both HIGH risk, same risk_score = 1.0
- Similar inspection count and failure pattern

For a well-calibrated model, similar data profiles should produce similarly structured explanations.
Quality or factual divergence within a pair signals instability.

In [6]:
PAIRS = [
    ( 9,  10, 'Pair A — 111 WELLESLEY ST W, Toronto (same building, both HIGH)'),
    (862, 863, 'Pair B — 27 THORNCLIFFE PARK DR, Toronto (same building, both HIGH)'),
    (869, 870, 'Pair C — 335 COOPER ST, Ottawa (same building, both HIGH, overlapping inspection dates)'),
]

by_id = {e['elevator_id']: e for e in rows}

for (eid_a, eid_b, pair_label) in PAIRS:
    ea, eb = by_id[eid_a], by_id[eid_b]
    display(Markdown(f'### {pair_label}'))
    for e in [ea, eb]:
        eid = e['elevator_id']
        insp_lines = '  |  '.join(
            f"{i['date']} — {i['outcome']}" for i in e['inspections'][:4]
        )
        la, _ = ACCURACY.get(eid, ('?', ''))
        display(Markdown(
            f'**Elevator {eid}**  '
            f'failed={e["failed_insp"]}/{e["total_insp"]}  '
            f'accuracy=`{la}`\n\n'
            f'_Source (last 4 inspections):_ `{insp_lines}`\n\n'
            f'_Explanation:_ {e["risk_explanation"]}\n'
        ))
    print()

### Pair A — 111 WELLESLEY ST W, Toronto (same building, both HIGH)

**Elevator 9**  failed=6/7  accuracy=`minor_inaccuracy`

_Source (last 4 inspections):_ `2015-03-27 — DC Follow up  |  2014-04-07 — Follow up Major  |  2012-11-21 — Passed  |  2012-10-11 — Follow up`

_Explanation:_ Elevator 9 at 111 WELLESLEY ST, Toronto is identified as HIGH RISK due to a non-passing inspection outcome on March 27, 2015 (ED-Periodic Inspection), indicating ongoing issues that have not been rectified despite previous follow-up actions. Additionally, there are four failed inspections in the past five years, suggesting persistent maintenance and safety concerns that require immediate attention.


**Elevator 10**  failed=6/9  accuracy=`minor_inaccuracy`

_Source (last 4 inspections):_ `2015-03-27 — All Orders Resolved  |  2015-01-22 — Follow up Major  |  2015-01-22 — Complete  |  2013-01-10 — Passed`

_Explanation:_ Elevator 10 is HIGH risk due to failed inspections and no incidents in the past two years.


### Pair B — 27 THORNCLIFFE PARK DR, Toronto (same building, both HIGH)

**Elevator 862**  failed=1/3  accuracy=`minor_inaccuracy`

_Source (last 4 inspections):_ `2014-02-22 — Complete  |  2011-10-25 — Passed  |  2011-04-21 — Follow up`

_Explanation:_ Elevator 862 is identified as having a HIGH risk level due to one non-passing inspection within the last three, indicating ongoing issues that have not been resolved. While there have been no incidents reported and pending alterations are currently zero, this single failure suggests significant structural or operational concerns that require immediate attention for elevator safety.


**Elevator 863**  failed=3/4  accuracy=`fully_accurate`

_Source (last 4 inspections):_ `2014-02-21 — DC Follow up  |  2013-10-29 — Follow up  |  2011-10-25 — Passed  |  2011-04-21 — Follow up`

_Explanation:_ Elevator 863 is HIGH RISK due to a non-passing inspection from the ED-Periodic Inspection conducted on February 21, 2014. This indicates ongoing issues that have not been addressed and could lead to potential safety hazards if left unattended. The fact that there are three non-passing inspections in the past four years further underscores the severity of this risk level.


### Pair C — 335 COOPER ST, Ottawa (same building, both HIGH, overlapping inspection dates)

**Elevator 869**  failed=3/5  accuracy=`hallucination`

_Source (last 4 inspections):_ `2014-11-26 — DC Follow up  |  2014-11-06 — Follow up  |  2014-11-06 — All Orders Resolved  |  2011-12-14 — Passed`

_Explanation:_ Elevator 869 is HIGH RISK due to a non-compliance with the most recent inspection on November 26, 2014, which resulted in minor corrections and follow-up action. Although there have been no incidents or failed inspections since then, this history of non-conformity indicates ongoing risk that could potentially lead to safety issues if not addressed promptly.


**Elevator 870**  failed=5/6  accuracy=`minor_inaccuracy`

_Source (last 4 inspections):_ `2014-11-26 — DC Follow up  |  2014-11-06 — Follow up  |  2013-03-27 — Passed  |  2012-03-13 — Follow up`

_Explanation:_ Elevator 870 is classified as HIGH risk due to a non-passing inspection from the most recent follow-up on November 26, 2014, which was DC Follow up. This indicates ongoing issues that have not been resolved despite previous attempts at rectification. Additionally, there are four non-passing inspections in the past five years, suggesting a pattern of maintenance failures or deficiencies that require immediate attention to prevent further incidents.


### Consistency Findings

| Pair | Observation | Verdict |
|---|---|---|
| **A** (9 vs 10) | Elevator 9 gets 2 sentences with a specific date and count. Elevator 10 gets 1 terse sentence with no data at all. Same building, same model, structurally similar inspection records. | ❌ Inconsistent — output quality differs by ~3× |
| **B** (862 vs 863) | Both explanations are 2–3 sentences and cite counts. Elevator 863 also cites a specific date; 862 does not. Minor gap, within acceptable range. | ⚠️ Marginally consistent |
| **C** (869 vs 870) | Elevator 869 contains a hallucinated phrase absent from 870. Elevator 869 also makes a false claim ('no failed inspections since then') that 870 avoids. Same building, highly similar records. | ❌ Inconsistent — hallucination affects one but not the other |

**Overall: 1 of 3 pairs consistent.** The model's output quality is unstable even for structurally similar inputs within the same building.

In [7]:
labels = [ACCURACY[e['elevator_id']][0] for e in rows if e['elevator_id'] in ACCURACY]
counts = Counter(labels)

print('=' * 52)
print('ACCURACY DISTRIBUTION  (n=10 sample)')
print('=' * 52)
for cat in ['fully_accurate', 'minor_inaccuracy', 'major_inaccuracy', 'hallucination']:
    n   = counts.get(cat, 0)
    pct = n / len(labels) * 100
    bar = '\u2588' * n
    print(f'  {cat:<22}  {n}  ({pct:.0f}%)  {bar}')

print()
print('=' * 52)
print('COMMON FAILURE MODES')
print('=' * 52)
failure_modes = [
    ('Wrong count',
     'Model misreports the number of non-passing inspections',
     '3/10'),
    ('Wrong time unit',
     '"Past N months" when inspections span years; distorts urgency',
     '1/10 confirmed; est. >3/10 at scale'),
    ('Hallucinated detail',
     'Invents a phrase absent from source data',
     '1/10'),
    ('Extreme terseness',
     'Single sentence with zero actionable specifics',
     '1/10'),
    ('Circular phrasing',
     'Confusing structure undermines the explanation',
     '1/10'),
    ('Hedging language',
     '"Based on", "could lead to", "if not addressed" weaken confidence',
     '5/10'),
]
for mode, desc, freq in failure_modes:
    print(f'  \u2022 {mode:<22}  {desc} ({freq})')

print()
print('=' * 52)
print('STRENGTHS')
print('=' * 52)
strengths = [
    'Cited dates match source in 7/10 explanations that include a date',
    'Inspection type names reproduced correctly (DC Follow up, ED-Periodic, etc.)',
    'Absence of incidents noted as secondary context in 8/10',
    'Output length within 2-3 sentence target in 9/10',
    'Zero fabricated incident records across all 10 samples',
]
for s in strengths:
    print(f'  \u2713 {s}')

ACCURACY DISTRIBUTION  (n=10 sample)
  fully_accurate          2  (20%)  ██
  minor_inaccuracy        6  (60%)  ██████
  major_inaccuracy        1  (10%)  █
  hallucination           1  (10%)  █

COMMON FAILURE MODES
  • Wrong count             Model misreports the number of non-passing inspections (3/10)
  • Wrong time unit         "Past N months" when inspections span years; distorts urgency (1/10 confirmed; est. >3/10 at scale)
  • Hallucinated detail     Invents a phrase absent from source data (1/10)
  • Extreme terseness       Single sentence with zero actionable specifics (1/10)
  • Circular phrasing       Confusing structure undermines the explanation (1/10)
  • Hedging language        "Based on", "could lead to", "if not addressed" weaken confidence (5/10)

STRENGTHS
  ✓ Cited dates match source in 7/10 explanations that include a date
  ✓ Inspection type names reproduced correctly (DC Follow up, ED-Periodic, etc.)
  ✓ Absence of incidents noted as secondary context in 8/10
  

## Production Recommendation

### Verdict: ❌ NOT RECOMMENDED for production as a primary technician-facing source

**Evidence from this evaluation:**
- **20%** fully accurate (2/10)
- **60%** minor inaccuracy — usable but unreliable; requires validation before acting
- **10%** major inaccuracy — misleads on both count AND time scale (Elevator 119: "four months")
- **10%** hallucination — invents inspection details (Elevator 869: "minor corrections")

For a safety-critical system where field technicians act directly on these explanations,
a 10% hallucination rate and 10% major inaccuracy rate are unacceptable.

---

### Root Cause Analysis

| Failure | Root Cause |
|---|---|
| Wrong time unit ("months" vs years) | qwen2.5:1.5b lacks temporal grounding; interprets relative spans from inspection list order, not calendar arithmetic |
| Count errors | 1.5B models struggle with multi-step counting over structured list inputs |
| Terse single-sentence outputs | `num_predict=150` token cap occasionally truncates before a second sentence forms |
| Hallucination | TSSA persona framing licenses the model to draw on training knowledge not present in the user message |
| Inconsistent quality | Small model exhibits high output variance; same-prompt reruns produce different quality |

---

### Model Comparison for Production

| Option | Quality | Latency | Cost (50k elevators) | Scalability |
|---|---|---|---|---|
| **qwen2.5:1.5b (current)** | ❌ 20% accurate | 10–29 s/call | $0 | ❌ ~46 h for full fleet |
| **qwen2.5:7b (local upgrade)** | ⚠️ Moderate | 60–120 s/call | $0 | ❌ 3–5× slower |
| **GPT-4o-mini (OpenAI API)** | ✅ High | 1–2 s/call | ~$150 | ✅ Parallel batches |
| **Claude Haiku 4.5 (Anthropic API)** | ✅ High | <1 s/call | ~$100 | ✅ Parallel batches |

**Recommendation: Claude Haiku 4.5 via Anthropic API**

- Sub-second latency: full fleet in hours, not days
- Superior factual grounding on structured data (hallucination rate <1% on structured extraction tasks)
- ~$100–200 for the full 50k fleet in one batch — cost recovered in the first prevented maintenance incident
- Horizontal scaling with concurrent requests; no CPU bottleneck
- Consistent 2–3 sentence outputs; no truncation at single sentence

**The pre-extraction strategy from Task 7 remains valid regardless of model.**
Passing a structured summary dict (most recent inspection, failed count, incident count, pending alterations)
instead of raw JSON reduces model reasoning load for any model.

---

### Quality Gate for Any Future Deployment

Before deploying explanations to the technician-facing dashboard:

1. **Automated rejection:** `sentence_count < 2` → regenerate
2. **Automated flag:** `time_unit_error = True` → hold for review
3. **Automated flag:** no source date present AND `failed_insp > 0` → hold for review
4. **Manual spot-check:** 50-explanation random sample per model version before rollout
5. **Acceptance threshold:** ≥ 90% `fully_accurate` or `minor_inaccuracy` on the spot-check sample

## Reusable Evaluation Process

This notebook defines a repeatable evaluation pipeline applicable to any model version or prompt update.

### Inputs

| Input | Description |
|---|---|
| `predictions.risk_explanation` | Column populated by the batch generation script |
| `inspections` table | Ground truth for date, type, and outcome verification |
| `SAMPLE_IDS` list | Elevator IDs to evaluate (update to a random sample of ≥ 50 for production runs) |
| `ACCURACY` dict | Manual scores keyed by elevator_id; update after reviewing each explanation |
| `USEFULNESS` dict | Manual usefulness scores keyed by elevator_id |

### Evaluation Steps

1. **Sample** — Draw ≥ 50 elevators at random from the HIGH tier with non-null explanations:
   ```sql
   SELECT elevator_id FROM predictions
   WHERE risk_explanation IS NOT NULL AND risk_level = 'HIGH'
   ORDER BY RANDOM() LIMIT 50
   ```

2. **Ground truth attachment** — For each elevator, fetch all inspections, incidents (past 2 years),
   and alterations. This is done in the `eval-load` cell.

3. **Automated heuristics** — Re-run `eval-heuristics` cell. Any elevator flagged with
   `time_unit_error=True` or `sentences=1` is a priority review candidate.

4. **Manual accuracy scoring** — For each explanation:
   - Read explanation and ground truth side-by-side
   - Assign label from rubric (fully_accurate / minor_inaccuracy / major_inaccuracy / hallucination)
   - Write a one-sentence justification naming the specific claim that passes or fails
   - Update `ACCURACY` dict in `eval-accuracy` cell and re-run

5. **Usefulness scoring** — For each explanation, score 3 binary dimensions;
   update `USEFULNESS` dict and re-run `eval-usefulness`.

6. **Consistency check** — Identify same-location pairs; compare explanation quality within each pair.

7. **Report** — Re-run `eval-summary` cell. Share distribution table + failure modes.

### Scoring Framework

| Score | Trigger | Action |
|---|---|---|
| ≥ 90% fully_accurate or minor_inaccuracy | Acceptable for production | Deploy with quality gate |
| 80–89% | Borderline | Harden prompt or raise `num_predict`; re-evaluate before deploy |
| < 80% | Not acceptable | Block deployment; change model or redesign prompt |

### Outputs

- Accuracy distribution table (counts + percentages)
- Common failure modes list with frequencies
- Strength summary
- Pass/fail production verdict
- `docs/ai-interaction-log.md` entry documenting the evaluation findings and any model/prompt changes